In [ ]:
import os
import json
from collections import defaultdict


In [ ]:
scores = ["proto_margin", "similarity", "soft_knn_margin_all", "soft_knn_margin_topk"]
IS_ZOO = True 
if IS_ZOO: 
    scores = ["soft_knn_margin_all", "soft_knn_margin_topk"]

model_variants = ["base", "ViTG"]
output_dir = "/sc/home/iven.schlegelmilch/bachelor_thesis_code/visualizations"

os.makedirs(output_dir, exist_ok=True)

file_pairs = defaultdict(dict)

print("Scanning directory and grouping files...")

for filename in os.listdir(output_dir):
    if not IS_ZOO and not filename.endswith(".json") or "PAIRED" in filename or "renamed_body" not in filename:
        continue
    if IS_ZOO and "renamed_body_subsampled_20pct" not in filename:
        continue
    for score in scores:
        if score in filename:
            for variant in model_variants:
                if variant in filename:
                    file_pairs[score][variant] = os.path.join(output_dir, filename)
                    print(f"  - Found '{variant}' file for '{score}': {filename}")
                    break
            break

    

In [ ]:
for score, paths in file_pairs.items():
    if "base" not in paths or "ViTG" not in paths:
        print(f"  - WARNING: Skipping '{score}'. Found only one variant, not a complete pair.")
        if "base" in paths: print(f"    - Found: {paths['base']}")
        if "ViTG" in paths: print(f"    - Found: {paths['ViTG']}")
        continue

    base_filepath = paths["base"]
    vitg_filepath = paths["ViTG"]

    try:
        with open(base_filepath, 'r') as f:
            base_data = json.load(f)
        with open(vitg_filepath, 'r') as f:
            vitg_data = json.load(f)
        print(f"\nProcessing pair for '{score}'...")
        print(f"  - Base file: {os.path.basename(base_filepath)}")
        print(f"  - ViTG file: {os.path.basename(vitg_filepath)}")
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"\nERROR: Could not read or parse files for '{score}'. Skipping. Error: {e}")
        continue

    intersection_results = {}
    common_keys = set(base_data.keys()) & set(vitg_data.keys())

    for key in common_keys:
        base_ids = base_data.get(key, [])
        vitg_ids = vitg_data.get(key, [])

        intersecting_ids = set(base_ids).intersection(set(vitg_ids))

        # Convert back to a sorted list for consistent output
        intersection_results[key] = sorted(list(intersecting_ids))
        
        print(f"    - Found {len(intersection_results[key])} intersecting items for '{key}' out of {len(base_ids)} (base) and {len(vitg_ids)} (ViTG)")

    if IS_ZOO:
        output_filename = f"{score}_PAIRED_intersection_zoo.json"
    else:
        output_filename = f"{score}_PAIRED_intersection.json"

    output_filepath = os.path.join(output_dir, output_filename)

    with open(output_filepath, 'w') as f:
        json.dump(intersection_results, f, indent=4)

    print(f"  -> Saved intersection results to: {output_filepath}")

print("\nIntersection process complete.")